# A.R.I.A. — Adaptive Road Intelligence Architecture
## 2-Stage Mixed Curriculum Training · YOLOv11n · 4-Class Type Detection

| Stage | Dataset | Purpose |
|---|---|---|
| 1 | RDD2022 all countries | Global foundation — learn 4 distinct defect types |
| 2 | IIT Madras + RDD2022 India | Rehearsal fine-tune (prevents catastrophic forgetting) |

**Output classes:** `0 longitudinal_crack` · `1 transverse_crack` · `2 alligator_crack` · `3 pothole`

In [ ]:
# %% [Cell 1] Install
!pip install -q ultralytics==8.4.21

In [ ]:
# %% [Cell 2] Imports, path discovery, directory tree
import os
import json
import shutil
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

import torch
from ultralytics import YOLO
from ultralytics import __version__ as ul_version

print(f"Ultralytics : {ul_version}")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA        : {torch.cuda.is_available()} — "
      f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

CONFIG_PATH = Path("/kaggle/working/aria_config.json")

def find_rdd_split(root="/kaggle/input") -> Path | None:
    for dp, dns, _ in os.walk(root):
        dl = {d.lower(): d for d in dns}
        if "rdd_split" in dl:
            return Path(dp) / dl["rdd_split"]
    return None

def find_itt_madras(root="/kaggle/input") -> Path | None:
    for dp, dns, fns in os.walk(root):
        if "data.yaml" not in fns:
            continue
        cand = Path(dp)
        cand_str = str(cand).lower()
        if "madras" not in cand_str and "itt" not in cand_str:
            continue
            
        img_dir = cand / "train" / "images"
        if img_dir.exists() and any(img_dir.iterdir()):
            return cand
    return None

RDD_SPLIT = find_rdd_split()
ITT_ROOT  = find_itt_madras()

print("\n" + "=" * 64)
print("/kaggle/input/  (2 levels)")
print("=" * 64)
for dp, dns, fns in os.walk("/kaggle/input"):
    depth = dp.replace("/kaggle/input", "").count(os.sep)
    if depth > 2:
        dns.clear()
        continue
    indent = "  " * depth
    print(f"{indent}{os.path.basename(dp) or 'input'}/  [{len(fns)} files]")

print(f"\nRDD_SPLIT  : {RDD_SPLIT}")
print(f"ITT Madras : {ITT_ROOT}")

if RDD_SPLIT is None or ITT_ROOT is None:
    raise RuntimeError(
        "One or both datasets not found. Check the tree above and update "
        "find_rdd_split / find_itt_madras if folder names differ."
    )

print("\n" + "=" * 64)
print("RDD2022 split counts")
print("=" * 64)
for sp in ("train", "val", "test"):
    img_d = RDD_SPLIT / sp / "images"
    lbl_d = RDD_SPLIT / sp / "labels"
    imgs = len(list(img_d.glob("*"))) if img_d.exists() else 0
    lbls = len(list(lbl_d.glob("*.txt"))) if lbl_d.exists() else 0
    print(f"  {sp:<6}: {imgs:>6} images   {lbls:>6} labels")

print("\nCountry prefixes in RDD2022/train/images/:")
pc = Counter()
for p in (RDD_SPLIT / "train" / "images").glob("*"):
    parts = p.stem.split("_")
    prefix = "_".join(x for x in parts if not x.isdigit())
    pc[prefix] += 1
for prefix, cnt in sorted(pc.items(), key=lambda x: -x[1]):
    print(f"  {prefix:<30}: {cnt:>5} images")

print(f"\nITT Madras split counts ({ITT_ROOT.name}):")
for sp in ("train", "valid", "test"):
    d = ITT_ROOT / sp / "images"
    cnt = len(list(d.glob("*"))) if d.exists() else 0
    print(f"  {sp:<6}: {cnt:>5} images")

cfg = json.loads(CONFIG_PATH.read_text()) if CONFIG_PATH.exists() else {}
cfg.update({"rdd_split": str(RDD_SPLIT), "itt_root": str(ITT_ROOT)})
CONFIG_PATH.write_text(json.dumps(cfg, indent=2))
print(f"\n✓ Paths saved to {CONFIG_PATH}")

In [ ]:
# %% [Cell 3] Stage 1 data preparation — ALL RDD2022 (4 Types)
import os
import json
import shutil
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

CONFIG_PATH = Path("/kaggle/working/aria_config.json")
cfg         = json.loads(CONFIG_PATH.read_text())
RDD_SPLIT   = Path(cfg["rdd_split"])               
STAGE1_DIR  = Path("/kaggle/working/stage1_data")  

shutil.rmtree(STAGE1_DIR, ignore_errors=True)

SPLIT_MAP = {
    RDD_SPLIT / "train": STAGE1_DIR / "train",
    RDD_SPLIT / "val":   STAGE1_DIR / "valid",
}
for dst in SPLIT_MAP.values():
    (dst / "images").mkdir(parents=True, exist_ok=True)
    (dst / "labels").mkdir(parents=True, exist_ok=True)

RDD_TO_TYPE = {0: 0, 1: 1, 2: 2, 3: 3, 4: 3}

cfg["rdd_to_type"] = RDD_TO_TYPE
CONFIG_PATH.write_text(json.dumps(cfg, indent=2))

def _scan_class_ids(lbl_dir: Path) -> set:
    ids = set()
    if not lbl_dir.exists():
        return ids
    for f in lbl_dir.glob("*.txt"):
        try:
            for line in f.read_text().strip().splitlines():
                if p := line.split():
                    ids.add(int(p[0]))
        except Exception:
            pass
    return ids

print("Scanning RDD2022 class IDs...")
all_ids: set = set()
for src in SPLIT_MAP:
    all_ids |= _scan_class_ids(src / "labels")
print(f"Original RDD2022 class IDs : {sorted(all_ids)}")
unknown_ids = all_ids - set(RDD_TO_TYPE.keys())
if unknown_ids:
    print(f"[WARN] Unknown class IDs: {sorted(unknown_ids)}")
print(f"Remapping → 0:longitudinal  1:transverse  2:alligator  3:pothole\n")

def _symlink_and_remap(task):
    img_src, dst_img, src_lbl, dst_lbl = task
    unknown_count = 0
    try:
        try:
            os.symlink(img_src, dst_img)
        except [FileExistsError, OSError]:
            pass 

        if src_lbl.exists():
            lines = []
            for line in src_lbl.read_text().strip().splitlines():
                p = line.strip().split()
                if len(p) >= 5:
                    orig_cls = int(p[0])
                    if orig_cls not in RDD_TO_TYPE:
                        unknown_count += 1
                        continue
                    new_cls = RDD_TO_TYPE[orig_cls]
                    lines.append(f"{new_cls} " + " ".join(p[1:]))
            dst_lbl.write_text("\n".join(lines))
        else:
            dst_lbl.write_text("")

        return True, None, unknown_count
    except Exception as e:
        return False, f"{img_src.name}: {e}", 0

total_ok = total_err = total_bg = total_unknown = 0
for src_split, dst_split in SPLIT_MAP.items():
    src_img_dir = src_split / "images"
    src_lbl_dir = src_split / "labels"
    if not src_img_dir.exists():
        raise FileNotFoundError(f"Images directory not found: {src_img_dir}")
        
    tasks = [
        (img,
         dst_split / "images" / img.name,
         src_lbl_dir / (img.stem + ".txt"),           
         dst_split / "labels" / (img.stem + ".txt")) 
        for img in src_img_dir.glob("*") if img.is_file()
    ]
    ok = err = unk = 0
    with ThreadPoolExecutor(max_workers=8) as pool:
        for success, msg, u in pool.map(_symlink_and_remap, tasks):
            if success:
                ok += 1
            else:
                err += 1
                if err <= 5: print(f"  [ERROR] {msg}")
            unk += u

    bg = sum(1 for f in (dst_split / "labels").glob("*.txt") if f.read_text().strip() == "")
    print(f"  {dst_split.name:<8}: {ok:>6} linked  {bg:>5} background  {unk:>4} dropped-cls  {err} errors")
    total_ok += ok; total_err += err; total_bg += bg; total_unknown += unk

if total_unknown > 0:
    print(f"\n[WARN] {total_unknown} labels had unknown RDD class IDs — dropped during map")

(STAGE1_DIR / "data.yaml").write_text(
    f"path: {STAGE1_DIR}\ntrain: train/images\nval: valid/images\n"
    "nc: 4\nnames: ['longitudinal_crack', 'transverse_crack', 'alligator_crack', 'pothole']\n"
)

train_n = len(list((STAGE1_DIR / "train" / "images").glob("*")))
valid_n = len(list((STAGE1_DIR / "valid" / "images").glob("*")))
print(f"\nStage 1 dataset ready:")
print(f"  train      : {train_n:>6} images")
print(f"  valid      : {valid_n:>6} images")
print(f"  background : {total_bg:>6} no-damage samples")

if train_n == 0:
    raise RuntimeError(f"0 images in stage1 train.")

cfg["stage1"] = {"train": train_n, "valid": valid_n, "background": total_bg}
CONFIG_PATH.write_text(json.dumps(cfg, indent=2))

In [ ]:
# %% [Cell 4] Stage 1 training — global road damage foundation (4 Types)
import json
import shutil
from pathlib import Path
from ultralytics import YOLO

CONFIG_PATH    = Path("/kaggle/working/aria_config.json")
STAGE1_DIR     = Path("/kaggle/working/stage1_data")
STAGE1_WEIGHTS = Path("/kaggle/working/aria_stage1.pt")
cfg = json.loads(CONFIG_PATH.read_text())

model   = YOLO("yolo11n.pt")
results = model.train(
    data          = str(STAGE1_DIR / "data.yaml"),
    epochs        = 50,
    imgsz         = 640,
    batch         = -1,       
    device        = 0,
    workers       = 4,
    name          = "aria_stage1",
    project       = "/kaggle/working/runs",
    exist_ok      = True,
    optimizer     = "SGD",
    lr0           = 0.01,
    momentum      = 0.937,
    weight_decay  = 0.0005,
    cos_lr        = True,
    warmup_epochs = 3,
    patience      = 10,       
    close_mosaic  = 10,
    fliplr        = 0.5,
    degrees       = 5.0,
    translate     = 0.1,
    scale         = 0.2,
    hsv_h         = 0.015,
    hsv_s         = 0.4,
    hsv_v         = 0.4,
    mixup         = 0.1,
    copy_paste    = 0.1,
    erasing       = 0.2,
    amp           = True,
    cache         = False,    
    plots         = True,
)

best_pt = Path(results.save_dir) / "weights" / "best.pt"
shutil.copy2(best_pt, STAGE1_WEIGHTS)

stage1_map50 = float(results.results_dict.get("metrics/mAP50(B)", 0.0))
cfg["stage1_mAP50"] = stage1_map50
CONFIG_PATH.write_text(json.dumps(cfg, indent=2))

print("\nCleaning up stage1_data to free disk...")
shutil.rmtree(STAGE1_DIR, ignore_errors=True)
print("  ✓ stage1_data removed")

print(f"\n{'─'*48}")
print(f"  Stage 1  mAP@50 : {stage1_map50:.4f}")
print(f"  Saved   → {STAGE1_WEIGHTS}")
print(f"{'─'*48}")

In [ ]:
# %% [Cell 5] Stage 2 data preparation — MIXED Train, PURE Val (4 Types)
import os
import json
import yaml
import shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

CONFIG_PATH = Path("/kaggle/working/aria_config.json")
cfg         = json.loads(CONFIG_PATH.read_text())
ITT_ROOT    = Path(cfg["itt_root"])
RDD_SPLIT   = Path(cfg["rdd_split"])
STAGE2_DIR  = Path("/kaggle/working/stage2_data")

shutil.rmtree(STAGE2_DIR, ignore_errors=True)

for sp in ("train", "valid"):
    (STAGE2_DIR / sp / "images").mkdir(parents=True, exist_ok=True)
    (STAGE2_DIR / sp / "labels").mkdir(parents=True, exist_ok=True)

itt_yaml  = yaml.safe_load((ITT_ROOT / "data.yaml").read_text())
itt_names = itt_yaml.get("names", [])
name_to_idx = {n.lower().strip(): i for i, n in enumerate(itt_names)}

_raw_mapping = cfg.get("rdd_to_type", {"0": 0, "1": 1, "2": 2, "3": 3, "4": 3})
RDD_TO_TYPE = {int(k): v for k, v in _raw_mapping.items()}

ITT_TO_TYPE = {}
type_rules = {
    "longitudinal crack": 0, 
    "crocodile crack": 2, 
    "pothole": 3
}
for expected, type_id in type_rules.items():
    if expected in name_to_idx:
        ITT_TO_TYPE[name_to_idx[expected]] = type_id

def _symlink_and_remap(task):
    img_src, dst_img, src_lbl, dst_lbl, mapping = task
    is_bg = False
    counts = {0:0, 1:0, 2:0, 3:0}
    try:
        try: os.symlink(img_src, dst_img)
        except [FileExistsError, OSError]: pass
            
        if src_lbl.exists():
            lines = []
            for line in src_lbl.read_text().strip().splitlines():
                if p := line.strip().split():
                    if len(p) >= 5 and int(p[0]) in mapping:
                        new_cls = mapping[int(p[0])]
                        counts[new_cls] += 1
                        lines.append(f"{new_cls} " + " ".join(p[1:]))
            dst_lbl.write_text("\n".join(lines))
        else:
            dst_lbl.write_text("")
            is_bg = True
        return True, None, is_bg, counts, "train" in str(dst_img)
    except Exception as e:
        return False, str(e), False, counts, False

all_tasks = []
itt_val_sp = "valid" if (ITT_ROOT / "valid" / "images").exists() else "val"

for sp_name, itt_sp in [("train", "train"), ("valid", itt_val_sp)]:
    src_split = ITT_ROOT / itt_sp
    if not (src_split / "images").exists(): continue
    dst_split = STAGE2_DIR / sp_name
    for img in (src_split / "images").glob("*"):
        if img.is_file():
            all_tasks.append((
                img, dst_split / "images" / img.name,
                (src_split / "labels") / (img.stem + ".txt"),
                dst_split / "labels" / (img.stem + ".txt"),
                ITT_TO_TYPE
            ))

src_train = RDD_SPLIT / "train"
if (src_train / "images").exists():
    india_imgs = [p for p in (src_train / "images").glob("*") if p.name.startswith("India_") and p.is_file()]
    dst_split = STAGE2_DIR / "train"
    for img in india_imgs:
        all_tasks.append((
            img, dst_split / "images" / img.name,
            (src_train / "labels") / (img.stem + ".txt"),
            dst_split / "labels" / (img.stem + ".txt"),
            RDD_TO_TYPE
        ))

print(f"Mixing IIT Madras + RDD2022 India (Train)...")
ok = err = total_bg = 0
train_counts = {0:0, 1:0, 2:0, 3:0}

with ThreadPoolExecutor(max_workers=8) as pool:
    for success, msg, is_bg, counts, is_train in pool.map(_symlink_and_remap, all_tasks):
        if success:
            ok += 1
            if is_bg: total_bg += 1
            if is_train:
                for k,v in counts.items(): train_counts[k] += v
        else:
            err += 1

(STAGE2_DIR / "data.yaml").write_text(
    f"path: {STAGE2_DIR}\ntrain: train/images\nval: valid/images\n"
    "nc: 4\nnames: ['longitudinal_crack', 'transverse_crack', 'alligator_crack', 'pothole']\n"
)

train_n = len(list((STAGE2_DIR / "train" / "images").glob("*")))
valid_n = len(list((STAGE2_DIR / "valid" / "images").glob("*")))

if train_n == 0: raise RuntimeError("0 images in stage2 train.")

print(f"\nStage 2 Dataset Ready (Train: Mixed | Val: IIT PURE):")
print(f"  train      : {train_n:>6} images")
print(f"  valid      : {valid_n:>6} images")

print(f"\n{'─'*40}")
print(f"  Stage 2 Train Class Distribution")
print(f"{'─'*40}")
print(f"  0 longitudinal_crack : {train_counts[0]:>6} instances")
print(f"  1 transverse_crack   : {train_counts[1]:>6} instances  *(RDD India only)*")
print(f"  2 alligator_crack    : {train_counts[2]:>6} instances")
print(f"  3 pothole            : {train_counts[3]:>6} instances")
print(f"{'─'*40}")

cfg["stage2_mixed"] = {"train": train_n, "valid": valid_n, "background": total_bg}
CONFIG_PATH.write_text(json.dumps(cfg, indent=2))

In [ ]:
# %% [Cell 6] Stage 2 training — MIXED Dataset Fine-Tuning
import json
import shutil
from pathlib import Path
from ultralytics import YOLO

CONFIG_PATH    = Path("/kaggle/working/aria_config.json")
STAGE2_DIR     = Path("/kaggle/working/stage2_data")
STAGE1_WEIGHTS = Path("/kaggle/working/aria_stage1.pt")
FINAL_WEIGHTS  = Path("/kaggle/working/aria_best_v1.pt")
cfg = json.loads(CONFIG_PATH.read_text())

if not STAGE1_WEIGHTS.exists():
    raise FileNotFoundError(f"Stage 1 weights not found: {STAGE1_WEIGHTS}. Run Cell 4 first.")

model = YOLO(str(STAGE1_WEIGHTS))
results = model.train(
    data          = str(STAGE2_DIR / "data.yaml"),
    epochs        = 40,         
    imgsz         = 640,
    batch         = -1,       
    device        = 0,
    workers       = 4,
    name          = "aria_stage2_mixed",
    project       = "/kaggle/working/runs",
    exist_ok      = True,
    
    freeze        = 3,          
    optimizer     = "AdamW",    
    lr0           = 0.001,      
    weight_decay  = 0.0005,
    cls           = 1.5,        
    
    warmup_epochs = 0.0,
    lrf           = 0.1,

    cos_lr        = True,
    close_mosaic  = 10,
    patience      = 15,         
    
    fliplr        = 0.5,
    degrees       = 5.0,
    translate     = 0.1,
    scale         = 0.2,
    hsv_h         = 0.015,
    hsv_s         = 0.4,
    hsv_v         = 0.4,
    
    mixup         = 0.0,
    copy_paste    = 0.0,
    erasing       = 0.0,
    
    amp           = True,
    cache         = False,      
    plots         = True,
)

best_pt = Path(results.save_dir) / "weights" / "best.pt"
shutil.copy2(best_pt, FINAL_WEIGHTS)

stage2_mixed_map50 = float(results.results_dict.get("metrics/mAP50(B)", 0.0))
cfg["stage2_mixed_mAP50"] = stage2_mixed_map50
CONFIG_PATH.write_text(json.dumps(cfg, indent=2))

print("\nCleaning up stage2_data/train to free disk...")
shutil.rmtree(STAGE2_DIR / "train", ignore_errors=True)
print("  ✓ stage2_data/train removed (valid/ preserved)")

print(f"\n{'─'*48}")
print(f"  Stage 2 (MIXED) mAP@50 : {stage2_mixed_map50:.4f}")
print(f"  Saved   → {FINAL_WEIGHTS}")
print(f"{'─'*48}")

In [ ]:
# %% [Cell 7] Final validation — IIT Madras val + RDD2022 test
import os
import json
import math
import shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from ultralytics import YOLO

CONFIG_PATH   = Path("/kaggle/working/aria_config.json")
FINAL_WEIGHTS = Path("/kaggle/working/aria_best_v1.pt")
cfg = json.loads(CONFIG_PATH.read_text())
RDD_SPLIT = Path(cfg["rdd_split"])

_raw_mapping = cfg.get("rdd_to_type", {"0": 0, "1": 1, "2": 2, "3": 3, "4": 3})
RDD_TO_TYPE = {int(k): v for k, v in _raw_mapping.items()}

rdd_test_map50   = float("nan")
rdd_test_map5095 = float("nan")

if not FINAL_WEIGHTS.exists():
    raise FileNotFoundError(f"Final weights not found: {FINAL_WEIGHTS}. Run Cell 6 first.")

model = YOLO(str(FINAL_WEIGHTS))

print("=" * 66)
print("  Validating on IIT Madras val set (training val set)")
print("=" * 66)

metrics_itt = model.val(data="/kaggle/working/stage2_data/data.yaml", device=0, imgsz=640, plots=True)
itt_map50   = float(metrics_itt.box.map50)
itt_map5095 = float(metrics_itt.box.map)

rdd_test_img = RDD_SPLIT / "test" / "images"
rdd_test_lbl = RDD_SPLIT / "test" / "labels"

if rdd_test_img.exists() and any(rdd_test_img.iterdir()):
    TEST_DIR = Path("/kaggle/working/rdd_test_eval")
    (TEST_DIR / "images").mkdir(parents=True, exist_ok=True)
    (TEST_DIR / "labels").mkdir(parents=True, exist_ok=True)

    def _prep_test_img(img):
        dst_img = TEST_DIR / "images" / img.name
        dst_lbl = TEST_DIR / "labels" / (img.stem + ".txt")
        try: os.symlink(img, dst_img)
        except OSError: pass 
        
        src_lbl = rdd_test_lbl / (img.stem + ".txt")
        if src_lbl.exists():
            lines = []
            for line in src_lbl.read_text().strip().splitlines():
                if p := line.strip().split():
                    if len(p) >= 5 and int(p[0]) in RDD_TO_TYPE:
                        new_cls = RDD_TO_TYPE[int(p[0])]
                        lines.append(f"{new_cls} " + " ".join(p[1:]))
            dst_lbl.write_text("\n".join(lines))
        else:
            dst_lbl.write_text("")
        return True

    test_imgs = [img for img in rdd_test_img.glob("*") if img.is_file()]
    with ThreadPoolExecutor(max_workers=8) as pool:
        list(pool.map(_prep_test_img, test_imgs))

    (TEST_DIR / "data.yaml").write_text(
        f"path: {TEST_DIR}\ntrain: images\nval: images\n"
        "nc: 4\nnames: ['longitudinal_crack', 'transverse_crack', 'alligator_crack', 'pothole']\n"
    )

    print(f"\n{'=' * 66}")
    print(f"  Validating on RDD2022 TEST split ({len(test_imgs)} images — truly held-out)")
    print(f"{'=' * 66}")

    try:
        metrics_rdd = model.val(data=str(TEST_DIR / "data.yaml"), device=0, imgsz=640, plots=True)
        rdd_test_map50   = float(metrics_rdd.box.map50)
        rdd_test_map5095 = float(metrics_rdd.box.map)
    finally:
        shutil.rmtree(TEST_DIR, ignore_errors=True)

cfg["final_itt_mAP50"]     = itt_map50
cfg["final_rdd_test_mAP50"]   = rdd_test_map50
CONFIG_PATH.write_text(json.dumps(cfg, indent=2))

s1 = cfg.get("stage1_mAP50", float("nan"))
s3 = cfg.get("stage2_mixed_mAP50", float("nan"))

W = 70
print(f"\n{'═'*W}")
print(f"  A.R.I.A.  Road Damage Detection — v3.0")
print(f"  Architecture : YOLOv11n (4-Class Type Detection)")
print(f"  Model        : {FINAL_WEIGHTS.name}")
print(f"{'─'*W}")
print(f"  IIT Madras val set (training val — optimistic)")
print(f"    mAP@50    : {itt_map50:.4f}")
print(f"{'─'*W}")
if not math.isnan(rdd_test_map50):
    print(f"  RDD2022 test split (held-out — TRUE generalization)")
    print(f"    mAP@50    : {rdd_test_map50:.4f}")
print(f"{'═'*W}")